# OpenAlex topics tests

## Import

In [11]:
import os
from pathlib import Path 
import pandas as pd
from sindex.sources.openalex.client import fetch_all_openalex_topics
from sindex.sources.openalex.jobs import get_primary_topic_for_doi

## Paths

In [6]:
current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, os.pardir))
topics_path = os.path.join(parent_dir, "input", "openalex_topics")
print(topics_path)

C:\Users\Admin\Documents\GitHub\sindex-pipeline\input\openalex_topics


## Get all topics (~4,500)

In [8]:
topics = fetch_all_openalex_topics()
len(topics)

4516

In [9]:
display(topics[0])

{'id': 'https://openalex.org/T10346',
 'display_name': 'Magnetic confinement fusion research',
 'description': 'This cluster of papers covers a wide range of topics in plasma physics and fusion research, including turbulence, transport, MHD stability, edge localized modes, zonal flows, confinement, neoclassical tearing modes, energetic particles, and diagnostics. The research spans from theoretical modeling to experimental observations in various fusion devices.',
 'keywords': ['Turbulence',
  'Tokamak',
  'Transport',
  'MHD Stability',
  'Edge Localized Modes',
  'Zonal Flows',
  'Confinement',
  'Neoclassical Tearing Modes',
  'Energetic Particles',
  'Diagnostics'],
 'ids': {'openalex': 'https://openalex.org/T10346',
  'wikipedia': 'https://en.wikipedia.org/wiki/Plasma_physics'},
 'subfield': {'id': 'https://openalex.org/subfields/3106',
  'display_name': 'Nuclear and High Energy Physics'},
 'field': {'id': 'https://openalex.org/fields/31',
  'display_name': 'Physics and Astronomy'

In [12]:
def write_topics_ndjson(topics: list[dict], out_path: str) -> str:
    p = Path(out_path)
    p.parent.mkdir(parents=True, exist_ok=True)
    with p.open("w", encoding="utf-8") as f:
        for t in topics:
            f.write(json.dumps(t, ensure_ascii=False) + "\n")
    return str(p)

output_path = os.path.join(topics_path, "topics.ndjson")
write_topics_ndjson(topics, output_path)

'C:\\Users\\Admin\\Documents\\GitHub\\sindex-pipeline\\input\\openalex_topics\\topics.ndjson'

## Analyze topics

In [14]:
def topics_to_table(topics: list[dict]) -> pd.DataFrame:
    rows = []
    for t in topics:
        rows.append({
            "topic_id": t.get("id"),
            "topic_name": t.get("display_name"),
            "subfield_name": t.get("subfield", {}).get("display_name"),
            "field_name": t.get("field", {}).get("display_name"),
            "domain_name": t.get("domain", {}).get("display_name"),
        })

    return pd.DataFrame(rows)

df = topics_to_table(topics)
display(df.head())

,topic_id,topic_name,subfield_name,field_name,domain_name
0,https://openalex.org/T10346,Magnetic confinement fusion research,Nuclear and High Energy Physics,Physics and Astronomy,Physical Sciences
1,https://openalex.org/T12157,Geochemistry and Geologic Mapping,Artificial Intelligence,Computer Science,Physical Sciences
2,https://openalex.org/T10451,Mycorrhizal Fungi and Plant Interactions,Plant Science,Agricultural and Biological Sciences,Life Sciences
3,https://openalex.org/T13370,Diverse Scientific and Economic Studies,Economics and Econometrics,"Economics, Econometrics and Finance",Social Sciences
4,https://openalex.org/T14423,Military Technology and Strategies,Aerospace Engineering,Engineering,Physical Sciences


In [15]:
df[[
    "topic_name",
    "subfield_name",
    "field_name",
    "domain_name",
]].nunique()

topic_name       4516
subfield_name     244
field_name         26
domain_name         4
dtype: int64

In [16]:
output_path = os.path.join(topics_path, "topics.csv")
df.to_csv(output_path, index=False)

## Get primary topic for a dataset

In [2]:
good_doi = "10.13026/kpb9-mt58"
get_primary_topic_for_doi(good_doi)

{'doi': '10.13026/kpb9-mt58',
 'work_id': 'https://openalex.org/W6960062145',
 'topic_id': 'https://openalex.org/T13154',
 'topic_name': 'Legal and Regulatory Analysis',
 'topic_score': 0.16790109872817993,
 'subfield_name': 'Transportation',
 'field_name': 'Social Sciences',
 'domain_name': 'Social Sciences'}

In [3]:
bad_doi = "10.dfsdf/kpb9-mt58"
get_primary_topic_for_doi(bad_doi)